In [ ]:
# Usando para suprimir mensagem de alertas jupyter_client
import warnings
# Suprime DeprecationWarning especificamente de jupyter_client
warnings.filterwarnings("ignore", category=DeprecationWarning, module='jupyter_client')
#########################################

Com o código acima, as mensagens de `DeprecationWarning` da biblioteca `jupyter_client` não serão mais exibidas na saída das células.

## Instalação e Importação da Biblioteca `mlxtend`

Para aplicar o algoritmo Apriori e gerar as regras de associação, utilizaremos a biblioteca `mlxtend` (Machine Learning Extensions). Primeiro, precisamos instalá-la e depois importar os módulos necessários.

In [29]:
# Instalar a biblioteca mlxtend
!pip install mlxtend

# Importar as funções necessárias da mlxtend
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

print("mlxtend instalada e importada com sucesso!")

mlxtend instalada e importada com sucesso!


# Mineração de Regras de Associação em Dados de Vendas

## 1. Introdução

### O que é Mineração de Dados?

Mineração de dados (ou Data Mining) é o processo de descobrir padrões, tendências e informações úteis a partir de grandes conjuntos de dados. É uma área multidisciplinar que usa técnicas de estatística, inteligência artificial e aprendizado de máquina para extrair conhecimento valioso que pode não ser óbvio à primeira vista.

### O que são Regras de Associação?

Regras de associação são técnicas de mineração de dados usadas para identificar relações entre conjuntos de itens em grandes bancos de dados. Elas expressam a probabilidade de que certos itens ocorram juntos em uma transação. O exemplo clássico é a regra "se um cliente compra fraldas, ele também tende a comprar cerveja".

### Onde essa Técnica é Aplicada?

As regras de associação são amplamente aplicadas em diversas áreas, incluindo:

*   **Varejo:** Análise de cestas de compras para otimizar layout de lojas, promoções e recomendações de produtos.
*   **Marketing:** Segmentação de clientes e personalização de campanhas.
*   **Saúde:** Identificação de sintomas que ocorrem juntos para diagnóstico de doenças.
*   **Web Mining:** Análise de padrões de navegação de usuários em websites.
*   **Telecomunicações:** Detecção de fraude ou identificação de padrões de uso de serviços.

### Por que ela é útil para análise de grandes volumes de dados?

Em um mundo onde as empresas geram e coletam vastas quantidades de dados diariamente, as regras de associação se tornam uma ferramenta poderosa para:

*   **Descobrir insights ocultos:** Revelar relações que seriam impossíveis de identificar manualmente.
*   **Melhorar a tomada de decisões:** Fornecer informações acionáveis para estratégias de negócios.
*   **Otimizar processos:** Aumentar a eficiência em diversas operações, desde a gestão de estoque até a personalização da experiência do cliente.
*   **Aumentar vendas e lucratividade:** Ao entender os padrões de compra, as empresas podem criar ofertas mais atraentes e eficazes.

## 2. Conceitos Principais

Para entender as regras de associação, é fundamental conhecer alguns termos:

*   **Itemset (Conjunto de Itens):** Uma coleção de um ou mais itens. Por exemplo, `{pão, leite}` é um itemset.

*   **Transação:** Um registro que contém um conjunto de itens. No contexto de um supermercado, uma transação seria uma única compra de um cliente, contendo todos os itens que ele adquiriu. Por exemplo, uma transação pode ser `{pão, leite, café}`.

*   **Suporte (Support):** Indica a frequência com que um itemset aparece no conjunto de dados. É a proporção de transações que contêm o itemset em relação ao número total de transações. Um suporte alto significa que o itemset é comum.
    *   **Fórmula:** `Support(A) = (Número de transações contendo A) / (Total de transações)`
    *   **Exemplo:** Se de 100 transações, 20 contêm `{pão, leite}`, o suporte de `{pão, leite}` é 20/100 = 0.2 (ou 20%).

*   **Confiança (Confidence):** Mede a força da implicação de uma regra. Para uma regra `A → B`, a confiança é a probabilidade de B ser comprado dado que A já foi comprado. Ou seja, quão frequentemente B aparece nas transações que já contêm A.
    *   **Fórmula:** `Confidence(A → B) = Support(A ∪ B) / Support(A)`
    *   **Exemplo:** Se o suporte de `{pão, leite}` é 0.2 e o suporte de `{pão}` é 0.25, então a confiança da regra `pão → leite` é 0.2 / 0.25 = 0.8 (ou 80%). Isso significa que, em 80% das vezes que um cliente compra pão, ele também compra leite.

*   **Regra no formato A → B:** Representa uma implicação, onde A é o **antecedente** (o que foi comprado) e B é o **consequente** (o que também foi comprado). Por exemplo, `fralda → cerveja` significa "se um cliente compra fraldas, então ele tende a comprar cerveja".

## 3. Criação do Conjunto de Dados Fictício

Vamos criar um conjunto de dados fictício que simule transações de vendas de um supermercado. Este conjunto de dados terá pelo menos 30 transações e incluirá alguns padrões de compra intencionais, que esperamos que o algoritmo de mineração de regras de associação consiga identificar.

In [20]:
# Importar bibliotecas necessárias
import pandas as pd
import random

# Lista de produtos disponíveis
produtos = [
    'pão', 'leite', 'café', 'manteiga', 'arroz', 'feijão', 'açúcar',
    'biscoito', 'fralda', 'cerveja', 'refrigerante', 'carne', 'queijo', 'presunto'
]

print("Produtos disponíveis:", produtos)

Produtos disponíveis: ['pão', 'leite', 'café', 'manteiga', 'arroz', 'feijão', 'açúcar', 'biscoito', 'fralda', 'cerveja', 'refrigerante', 'carne', 'queijo', 'presunto']


In [21]:
# Gerar transações com padrões intencionais

def gerar_transacao(produtos_base, num_min=2, num_max=5):
    return random.sample(produtos_base, random.randint(num_min, num_max))

transacoes_ficticias = []

# Padrões intencionais:
# pão + leite
# arroz + feijão
# fralda + cerveja
# queijo + presunto

# Gerar transações com padrões fortes
for _ in range(10): # 10 transações com 'pão' e 'leite'
    transacao = ['pão', 'leite'] + gerar_transacao([p for p in produtos if p not in ['pão', 'leite']], 1, 3)
    transacoes_ficticias.append(list(set(transacao))) # Remove duplicatas e garante lista

for _ in range(8): # 8 transações com 'arroz' e 'feijão'
    transacao = ['arroz', 'feijão'] + gerar_transacao([p for p in produtos if p not in ['arroz', 'feijão']], 1, 2)
    transacoes_ficticias.append(list(set(transacao)))

for _ in range(7): # 7 transações com 'fralda' e 'cerveja'
    transacao = ['fralda', 'cerveja'] + gerar_transacao([p for p in produtos if p not in ['fralda', 'cerveja']], 1, 2)
    transacoes_ficticias.append(list(set(transacao)))

for _ in range(6): # 6 transações com 'queijo', 'presunto'
    transacao = ['queijo', 'presunto'] + gerar_transacao([p for p in produtos if p not in ['queijo', 'presunto']], 1, 2)
    transacoes_ficticias.append(list(set(transacao)))

# Gerar transações aleatórias adicionais para atingir pelo menos 30 e introduzir variedade
while len(transacoes_ficticias) < 35: # Garantir pelo menos 35 transações no total
    transacoes_ficticias.append(gerar_transacao(produtos, 2, 6))

# Exibir as primeiras 10 transações para verificação
print(f"Total de transações geradas: {len(transacoes_ficticias)}")
print("\nPrimeiras 10 transações:")
for i, t in enumerate(transacoes_ficticias[:10]):
    print(f"Transação {i+1}: {t}")

Total de transações geradas: 35

Primeiras 10 transações:
Transação 1: ['arroz', 'leite', 'pão', 'feijão']
Transação 2: ['arroz', 'leite', 'pão', 'biscoito']
Transação 3: ['leite', 'pão', 'queijo', 'cerveja']
Transação 4: ['leite', 'pão', 'biscoito', 'queijo']
Transação 5: ['leite', 'pão', 'fralda', 'carne']
Transação 6: ['leite', 'pão', 'refrigerante', 'feijão']
Transação 7: ['pão', 'leite', 'fralda', 'presunto', 'carne']
Transação 8: ['presunto', 'leite', 'pão']
Transação 9: ['pão', 'queijo', 'arroz', 'leite', 'carne']
Transação 10: ['arroz', 'leite', 'pão', 'biscoito']


## 4. Preparação dos Dados: One-Hot Encoding

O algoritmo Apriori da biblioteca `mlxtend` espera que os dados de transação estejam em um formato específico: uma matriz booleana (ou DataFrame) onde cada linha representa uma transação e cada coluna representa um item. Um valor `True` ou `1` indica a presença do item naquela transação, e `False` ou `0` indica sua ausência.

Para isso, utilizaremos o `TransactionEncoder` da `mlxtend`, que converterá nossa lista de transações (lista de listas de itens) em um DataFrame one-hot encoded.

In [22]:
# Instanciar o TransactionEncoder
te = TransactionEncoder()

# Ajustar e transformar as transações para o formato one-hot encoded
te_ary = te.fit(transacoes_ficticias).transform(transacoes_ficticias)

# Converter o array resultante em um DataFrame pandas
df_transacoes_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print("Shape do DataFrame one-hot encoded:", df_transacoes_encoded.shape)
print("\nPrimeiras 5 linhas do DataFrame one-hot encoded:")
display(df_transacoes_encoded.head())

Shape do DataFrame one-hot encoded: (35, 14)

Primeiras 5 linhas do DataFrame one-hot encoded:


,arroz,açúcar,biscoito,café,carne,cerveja,feijão,fralda,leite,manteiga,presunto,pão,queijo,refrigerante
0,True,False,False,False,False,False,True,False,True,False,False,True,False,False
1,True,False,True,False,False,False,False,False,True,False,False,True,False,False
2,False,False,False,False,False,True,False,False,True,False,False,True,True,False
3,False,False,True,False,False,False,False,False,True,False,False,True,True,False
4,False,False,False,False,True,False,False,True,True,False,False,True,False,False


## 5. Aplicação do Algoritmo Apriori para Encontrar Itemsets Frequentes

O algoritmo Apriori é uma ferramenta essencial na mineração de regras de associação. Ele identifica "itemsets frequentes" em um conjunto de dados, que são coleções de itens que aparecem juntos com uma frequência mínima definida. Este processo é crucial porque as regras de associação são geradas a partir desses itemsets frequentes, e um passo eficiente para encontrar esses conjuntos é fundamental para a performance do algoritmo.

Para o nosso exemplo, utilizaremos a função `apriori` da biblioteca `mlxtend`. Precisamos definir um `min_support`, que é o suporte mínimo (em decimal) para que um itemset seja considerado frequente. Um valor muito baixo pode gerar muitos itemsets e regras irrelevantes, enquanto um valor muito alto pode fazer com que padrões importantes sejam perdidos. Vamos começar com um `min_support` de 0.2 (20%).

In [23]:
# Aplicar o algoritmo Apriori
frequent_itemsets = apriori(df_transacoes_encoded, min_support=0.2, use_colnames=True)

print("Itemsets Frequentes encontrados (com min_support=0.2):")
display(frequent_itemsets.head())

Itemsets Frequentes encontrados (com min_support=0.2):


,support,itemsets
0,0.428571,(arroz)
1,0.228571,(biscoito)
2,0.314286,(cerveja)
3,0.400000,(feijão)
4,0.314286,(fralda)


## 6. Geração de Regras de Associação

Com os itemsets frequentes em mãos, o próximo passo é gerar as regras de associação a partir deles. Para isso, utilizaremos a função `association_rules` da `mlxtend`. Esta função permite que especifiquemos uma métrica de avaliação (como `confidence`, `lift`, `leverage` ou `conviction`) e um limite mínimo para essa métrica.

Vamos usar a métrica `confidence` (confiança) com um `min_threshold` de 0.7 (70%). Isso significa que só nos interessam as regras onde a probabilidade de o consequente ocorrer, dado o antecedente, seja de pelo menos 70%.

In [28]:
# Gerar as regras de associação
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)

#Contar total de regras geradas
total_regras = len(rules)
print(f"Total de regras geradas: {total_regras}")

print("Regras de Associação geradas (com min_confidence=0.7):")
display(rules.head(10))

Total de regras geradas: 2
Regras de Associação geradas (com min_confidence=0.7):


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(feijão),(arroz),0.4,0.428571,0.285714,0.714286,1.666667,1.0,0.114286,2.0,0.666667,0.526316,0.5,0.690476
1,(leite),(pão),0.4,0.428571,0.285714,0.714286,1.666667,1.0,0.114286,2.0,0.666667,0.526316,0.5,0.690476


## 7. Análise das Regras de Associação

As regras geradas fornecem insights valiosos sobre os padrões de compra. Vamos analisar as colunas do DataFrame `rules`:

*   **antecedents:** Os itens no lado esquerdo da regra (o "se" da regra).
*   **consequents:** Os itens no lado direito da regra (o "então" da regra).
*   **antecedent support:** Suporte do antecedente (frequência do antecedente no total de transações).
*   **consequent support:** Suporte do consequente (frequência do consequente no total de transações).
*   **support:** Suporte da regra (frequência com que antecedente e consequente aparecem juntos).
*   **confidence:** Confiança da regra (probabilidade do consequente ocorrer, dado o antecedente).
*   **lift:** Mede o quão mais provável é que o consequente ocorra quando o antecedente está presente, em comparação com sua ocorrência independente. Um `lift` > 1 indica uma associação positiva, < 1 uma associação negativa e = 1 independência.
*   **leverage:** Semelhante ao lift, mede a diferença entre a frequência observada e esperada de antecedentes e consequentes aparecendo juntos.
*   **conviction:** Outra métrica que compara a probabilidade de o antecedente ocorrer sem o consequente com a probabilidade esperada se eles fossem independentes. Um valor alto indica forte dependência.

In [19]:
# Exibir as regras ordenadas por 'confidence' para ver as associações mais fortes
print("Regras de Associação ordenadas por Confidence (top 10):")
display(rules.sort_values(by='confidence', ascending=False).head(10))

Regras de Associação ordenadas por Confidence (top 10):


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(arroz),(feijão),0.285714,0.400000,0.257143,0.900000,2.250000,1.0,0.142857,6.000000,0.777778,0.600000,0.833333,0.771429
1,(leite),(pão),0.371429,0.457143,0.314286,0.846154,1.850962,1.0,0.144490,3.528571,0.731405,0.611111,0.716599,0.766827
